# LangChain Hands-on 03 — Output Parsers & Structured Output
**Trainer:** Ajeetkumar

**What we will practise**
- StrOutputParser
- CommaSeparatedList, NumberedList, MarkdownList
- JsonOutputParser
- PydanticOutputParser + format instructions
- XMLOutputParser
- OutputFixingParser (auto-fix bad output)
- `with_structured_output` — the modern, most reliable way

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-groq langchain-classic

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

# Why Output Parsers?
- LLM returns **text** → our code needs **list / dict / object**
- Parser = converts LLM text into Python data
- Use in apps: save to DB, show in UI, call APIs

In [ ]:
response = llm.invoke("Give me a short example of AI use in daily life?")

## 1. StrOutputParser — AIMessage → plain string

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
parser = StrOutputParser()
parser.parse(response.content)

In [ ]:
parser.invoke(response)       # works directly on the AIMessage

## 2. CommaSeparatedListOutputParser — text → Python list

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [ ]:
parser = CommaSeparatedListOutputParser()
parser.get_format_instructions()

In [ ]:
prompt = f"List five popular programming languages. {parser.get_format_instructions()}"
response = llm.invoke(prompt)
print(response.content)

In [ ]:
languages = parser.parse(response.content)
languages

In [ ]:
type(languages), len(languages)

## 3. NumberedListOutputParser & MarkdownListOutputParser

In [ ]:
from langchain_core.output_parsers import NumberedListOutputParser, MarkdownListOutputParser

In [ ]:
parser = NumberedListOutputParser()
response = llm.invoke(f"Give 4 tips to learn Python. {parser.get_format_instructions()}")
parser.parse(response.content)

In [ ]:
parser = MarkdownListOutputParser()
response = llm.invoke(f"Give 3 benefits of exercise. {parser.get_format_instructions()}")
parser.parse(response.content)

## 4. JsonOutputParser — text → dict

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt = PromptTemplate(
    input_variables=[],
    template="Provide details of a smartphone in JSON with keys: name, price, features. Only JSON."
)
response = llm.invoke(prompt.format())
print(response.content)

In [ ]:
parser = JsonOutputParser()
data = parser.parse(response.content)
data

In [ ]:
data["name"], data["price"]

## 5. PydanticOutputParser — text → validated Python object

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [ ]:
class Smartphone(BaseModel):
    name: str = Field(description="model name")
    price: int = Field(description="price in INR")
    features: list[str] = Field(description="top 3 features")

In [ ]:
parser = PydanticOutputParser(pydantic_object=Smartphone)
format_instructions = parser.get_format_instructions()
print(format_instructions)

In [ ]:
prompt = f"Provide the details of a popular smartphone. {format_instructions}"
response = llm.invoke(prompt)
print(response.content)

In [ ]:
phone = parser.parse(response.content)
phone

In [ ]:
phone.name, phone.price, phone.features

In [ ]:
phone.model_dump()

### Pydantic parser with a PromptTemplate (clean way)

In [ ]:
prompt = PromptTemplate(
    template="Extract the movie details.\n{format_instructions}\nText: {text}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [ ]:
class Movie(BaseModel):
    title: str
    year: int
    director: str
    genre: list[str]

movie_parser = PydanticOutputParser(pydantic_object=Movie)
prompt = prompt.partial(format_instructions=movie_parser.get_format_instructions())

In [ ]:
response = llm.invoke(prompt.format(text="3 Idiots is a 2009 Hindi comedy-drama directed by Rajkumar Hirani."))
movie_parser.parse(response.content)

## 6. XMLOutputParser

In [ ]:
from langchain_core.output_parsers import XMLOutputParser

In [ ]:
parser = XMLOutputParser(tags=["books", "book", "title", "author"])
response = llm.invoke(f"List 2 famous Indian novels with author. {parser.get_format_instructions()}")
print(response.content)

In [ ]:
parser.parse(response.content)

## 7. OutputFixingParser — auto-repair broken output using the LLM

In [ ]:
from langchain_classic.output_parsers import OutputFixingParser

In [ ]:
bad_output = "{'name': 'iPhone 16', 'price': 'eighty thousand', features: ['camera']}"   # invalid JSON + wrong type

In [ ]:
# parser.parse(bad_output)   # ❌ this would raise an error

In [ ]:
fixing_parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=Smartphone), llm=llm)
fixing_parser.parse(bad_output)

## 8. ⭐ with_structured_output — the modern way (recommended)
- Uses the model's **native tool calling / JSON mode**
- No format instructions needed, no parsing errors
- Returns a Pydantic object directly

In [ ]:
structured_llm = llm.with_structured_output(Smartphone)
phone = structured_llm.invoke("Tell me about the Samsung Galaxy S24")
phone

In [ ]:
type(phone)

### Real use case — fraud check from a bank SMS

In [ ]:
from typing import Literal

class TransactionRisk(BaseModel):
    amount_inr: float
    merchant: str
    channel: Literal["UPI", "card", "netbanking", "unknown"]
    risk_level: Literal["LOW", "MEDIUM", "HIGH"]
    reasons: list[str]

In [ ]:
risk_llm = llm.with_structured_output(TransactionRisk)
result = risk_llm.invoke("UPI payment of Rs 49,999 to 'QuickLoanzz' at 3:12 AM from a new device, right after a password reset.")
result

In [ ]:
print(result.risk_level, "|", result.reasons)

### Also works with TypedDict / JSON schema (returns dict)

In [ ]:
from typing_extensions import TypedDict, Annotated

class Resume(TypedDict):
    name: Annotated[str, "candidate name"]
    skills: Annotated[list[str], "technical skills"]
    experience_years: Annotated[int, "total years of experience"]

In [ ]:
resume_llm = llm.with_structured_output(Resume)
resume_llm.invoke("Ajeet is an AI engineer with 5 years of experience in Python, LangChain, Azure and fraud analytics.")

# Which one should I use?
| Parser | Output | Use when |
|---|---|---|
| `StrOutputParser` | str | chat answers, chains |
| `CommaSeparatedList` / `NumberedList` | list | simple lists |
| `JsonOutputParser` | dict | quick JSON, flexible keys |
| `PydanticOutputParser` | object | validation needed, model without tool calling |
| **`with_structured_output`** | object / dict | **best default for modern models** |
| `OutputFixingParser` | object | repair broken outputs |

## 🧪 Practice
1. Extract `name, email, phone` from a job application text with `with_structured_output`.
2. Get 10 Indian cities as a Python list with `CommaSeparatedListOutputParser`.
3. Make a `Recipe(name, ingredients: list[str], minutes: int)` Pydantic parser.